# M5 Store-Level Ensemble · Chronos-2 · v1

Trains **one Chronos-2 model per M5 store** (CA_1 … WI_3).
Each model is fit exclusively on its store's items — no cross-store pooling.
The final ensemble is the concatenation of 10 store-specific forecasts
(equal-weight average per item, which is trivial since each item belongs
to exactly one store).

Parallelism: stores run two-at-a-time (one per GPU) via `ThreadPoolExecutor`.
Store DataFrames are pre-filtered on CPU threads before GPU work starts.
Memory guards: psutil RAM-cap check + `gc.collect(2)` + `malloc_trim` after each store.

**Workflow**
1. Run all stores with `ft_steps=0` → zeroshot baselines cached.
2. Inspect per-store WRMSSE. Set `ft_steps>0` for stores that could benefit.
3. Re-run — only changed stores recompute; rest are instant cache hits.

# 1 · Imports

In [ ]:
import os
import sys
sys.path.append("/home/nmwamsojo/tsfm-explo/src/jobs/")

import gc
import time
import ctypes
import contextlib
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import psutil
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

from m5_dataprep import M5DataPipeline
from m5_exploration import M5ExplorationSuite, DEFAULT_CHRONOS_CONFIG
from m5_evaluator import M5Evaluator

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({props.total_memory // 1024**3} GB)")

In [ ]:
# 4 threads per GPU worker × 2 workers = 8 of 20 CPUs for BLAS/data loading.
# 4 additional CPU prep threads stay in the same budget.
os.environ["OMP_NUM_THREADS"]        = "4"
os.environ["MKL_NUM_THREADS"]        = "4"
os.environ["OPENBLAS_NUM_THREADS"]   = "4"
os.environ["VECLIB_MAXIMUM_THREADS"] = "4"
os.environ["NUMEXPR_NUM_THREADS"]    = "4"
os.environ["CUDA_VISIBLE_DEVICES"]   = "0,1"

DEVICES = [f"cuda:{i}" for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else ["cpu"]
N_GPUS  = len(DEVICES)
print(f"Active devices ({N_GPUS}): {DEVICES}")

if torch.cuda.is_available():
    for _d in DEVICES:
        torch.zeros(1, device=_d)
    torch.cuda.synchronize()
    print("CUDA contexts pre-initialized on all devices.")

# 2 · Config
**Edit this cell only** to change experiment settings.

In [ ]:
# ── Data paths ────────────────────────────────────────────────────────────────
DATA_PATH     = "/mnt/lab/datasets/M5/jointed_M5.parquet"
CALENDAR_PATH = "/mnt/lab/nmwamsojo/m5_data/calendar.csv"
ACTUALS_PATH  = "/mnt/lab/nmwamsojo/m5_data/sales_test_evaluation.csv"

DATA_TAG   = "sales_only"
CUTOFF_DAY = (
    pd.to_datetime("2016-05-22") - pd.Timedelta(days=28)
).strftime("%Y-%m-%d")
print(f"Cutoff day : {CUTOFF_DAY}")
print(f"Data tag   : {DATA_TAG}")

# ── Global execution caps ─────────────────────────────────────────────────────
RAM_CAP_GB       = 38.0   # pause worker if process RSS exceeds this
RAM_RETRY_SLEEP  = 60     # seconds between RAM retries
RAM_MAX_RETRIES  = 3      # max retries before skipping a store
MAX_CONCURRENT   = 2      # GPU workers (one per GPU)
CPU_PREP_WORKERS = 4      # CPU threads for DataFrame pre-filtering

# ── Store list (order sets GPU assignment: even-index→cuda:0, odd→cuda:1) ─────
STORES = ["CA_1", "CA_2", "CA_3", "CA_4",
          "TX_1", "TX_2", "TX_3",
          "WI_1", "WI_2", "WI_3"]

# ── Per-store model configs ───────────────────────────────────────────────────
# ft_steps=0  → zeroshot (tag gets hpo_zs_ prefix; cached and reused).
# ft_steps>0  → finetuned (FT params below apply; cached by full tag).
# Workflow: run all at ft_steps=0 first, inspect per-store WRMSSE,
# then set ft_steps>0 for stores that benefit — others stay cached.
STORE_CONFIGS: dict[str, dict] = {
    "CA_1": {"ft_steps": 0, "ft_mode": "lora", "ft_lr": 1e-4, "ft_bs": 64,
             "context_length": 256, "use_is_weekend": False, "use_event": False,
             "use_price": False, "use_snap": True},
    "CA_2": {"ft_steps": 0, "ft_mode": "lora", "ft_lr": 1e-4, "ft_bs": 64,
             "context_length": 256, "use_is_weekend": False, "use_event": False,
             "use_price": False, "use_snap": True},
    "CA_3": {"ft_steps": 0, "ft_mode": "lora", "ft_lr": 1e-4, "ft_bs": 64,
             "context_length": 256, "use_is_weekend": False, "use_event": False,
             "use_price": False, "use_snap": True},
    "CA_4": {"ft_steps": 0, "ft_mode": "lora", "ft_lr": 1e-4, "ft_bs": 64,
             "context_length": 256, "use_is_weekend": False, "use_event": False,
             "use_price": False, "use_snap": True},
    "TX_1": {"ft_steps": 0, "ft_mode": "lora", "ft_lr": 1e-4, "ft_bs": 64,
             "context_length": 256, "use_is_weekend": False, "use_event": False,
             "use_price": False, "use_snap": True},
    "TX_2": {"ft_steps": 0, "ft_mode": "lora", "ft_lr": 1e-4, "ft_bs": 64,
             "context_length": 256, "use_is_weekend": False, "use_event": False,
             "use_price": False, "use_snap": True},
    "TX_3": {"ft_steps": 0, "ft_mode": "lora", "ft_lr": 1e-4, "ft_bs": 64,
             "context_length": 256, "use_is_weekend": False, "use_event": False,
             "use_price": False, "use_snap": True},
    "WI_1": {"ft_steps": 0, "ft_mode": "lora", "ft_lr": 1e-4, "ft_bs": 64,
             "context_length": 256, "use_is_weekend": False, "use_event": False,
             "use_price": False, "use_snap": True},
    "WI_2": {"ft_steps": 0, "ft_mode": "lora", "ft_lr": 1e-4, "ft_bs": 64,
             "context_length": 256, "use_is_weekend": False, "use_event": False,
             "use_price": False, "use_snap": True},
    "WI_3": {"ft_steps": 0, "ft_mode": "lora", "ft_lr": 1e-4, "ft_bs": 64,
             "context_length": 256, "use_is_weekend": False, "use_event": False,
             "use_price": False, "use_snap": True},
}
assert set(STORES) == set(STORE_CONFIGS.keys()), (
    f"STORES and STORE_CONFIGS keys differ: {set(STORES).symmetric_difference(set(STORE_CONFIGS.keys()))}"
)

# ── AutoGluon wrapper ─────────────────────────────────────────────────────────
BATCH_SIZE = 64
WRAPPER = {
    "eval_metric":          "RMSSE",
    "enable_ensemble":      False,
    "skip_model_selection": True,
    "verbosity":            1,
}
CFG_CHRONOS_BASE = {
    **DEFAULT_CHRONOS_CONFIG,
    "use_static": False,
}

# ── Covariate column lookup (identical to v7) ──────────────────────────────────
COVARIATE_OPTIONS = {
    "none":                        [],
    "is_weekend":                  ["is_friday", "is_saturday", "is_sunday"],
    "event":                       ["event_name_1", "event_type_1"],
    "price":                       ["sell_price"],
    "snap":                        ["snap_CA", "snap_TX", "snap_WI"],
    "is_weekend_event":            ["is_friday", "is_saturday", "is_sunday",
                                    "event_name_1", "event_type_1"],
    "is_weekend_price":            ["is_friday", "is_saturday", "is_sunday", "sell_price"],
    "is_weekend_snap":             ["is_friday", "is_saturday", "is_sunday",
                                    "snap_CA", "snap_TX", "snap_WI"],
    "event_price":                 ["event_name_1", "event_type_1", "sell_price"],
    "event_snap":                  ["event_name_1", "event_type_1",
                                    "snap_CA", "snap_TX", "snap_WI"],
    "price_snap":                  ["sell_price", "snap_CA", "snap_TX", "snap_WI"],
    "is_weekend_event_price":      ["is_friday", "is_saturday", "is_sunday",
                                    "event_name_1", "event_type_1", "sell_price"],
    "is_weekend_event_snap":       ["is_friday", "is_saturday", "is_sunday",
                                    "event_name_1", "event_type_1",
                                    "snap_CA", "snap_TX", "snap_WI"],
    "is_weekend_price_snap":       ["is_friday", "is_saturday", "is_sunday",
                                    "sell_price", "snap_CA", "snap_TX", "snap_WI"],
    "event_price_snap":            ["event_name_1", "event_type_1", "sell_price",
                                    "snap_CA", "snap_TX", "snap_WI"],
    "is_weekend_event_price_snap": ["is_friday", "is_saturday", "is_sunday",
                                    "event_name_1", "event_type_1", "sell_price",
                                    "snap_CA", "snap_TX", "snap_WI"],
}

print(f"Stores           : {STORES}")
print(f"RAM cap          : {RAM_CAP_GB} GB")
print(f"GPU workers      : {MAX_CONCURRENT}")
print(f"CPU prep threads : {CPU_PREP_WORKERS}")
print(f"Covariate combos : {len(COVARIATE_OPTIONS)}")

# 3 · Data

In [ ]:
pipeline = M5DataPipeline(config={"tag": DATA_TAG})
hist_df, hist_df_trimmed, future_df, static_df, weights_scales = (
    pipeline.get_prepared_data(DATA_PATH, CUTOFF_DAY, level=12, force_reprepare=False)
)

print(f"\nhist_df         : {hist_df.shape}  cols: {list(hist_df.columns)}")
print(f"hist_df_trimmed : {hist_df_trimmed.shape}")
print(f"future_df       : {future_df.shape}")
print(f"static_df       : {static_df.shape}")
print(f"weights_scales  : {weights_scales.shape}")

del pipeline
gc.collect()

In [ ]:
def _wide_to_long(gt_wide: pd.DataFrame, calendar_path: str) -> pd.DataFrame:
    if "id" not in gt_wide.columns:
        gt_wide["id"] = gt_wide["item_id"] + "_" + gt_wide["store_id"] + "_evaluation"
    day_cols = [c for c in gt_wide.columns if c.startswith("d_")]
    long = gt_wide.melt(id_vars=["id"], value_vars=day_cols,
                        var_name="d", value_name="sales_quantity")
    cal = pd.read_csv(calendar_path, usecols=["d", "date"])
    cal["date"] = pd.to_datetime(cal["date"])
    long = long.merge(cal, on="d", how="left").drop(columns=["d"])
    long["id"] = long["id"].str.replace("_evaluation", "", regex=False)
    return long[["id", "date", "sales_quantity"]]


if CUTOFF_DAY == "2016-05-22":
    _eval_raw = pd.read_csv(ACTUALS_PATH)
    df_actual = _wide_to_long(_eval_raw, CALENDAR_PATH)
    del _eval_raw
else:
    df_actual = (
        pd.read_parquet(DATA_PATH, columns=["id", "date", "sold"])
        .rename(columns={"sold": "sales_quantity"})
    )
    df_actual["id"] = (
        df_actual["id"].astype(str)
        .str.replace("_evaluation", "", regex=False)
        .str.replace("_validation",  "", regex=False)
    )

print(f"Actuals : {df_actual.shape}  |  "
      f"{df_actual['date'].min().date()} \u2192 {df_actual['date'].max().date()}")

# 4 · Evaluator & Suite
One `M5ExplorationSuite` per GPU — each suite's TSDF cache holds at most
one entry and evicts automatically when the store changes.

In [ ]:
evaluator = M5Evaluator(
    raw_train_df     = hist_df,
    trimmed_train_df = hist_df_trimmed,
    static_df        = static_df,
    weights_df       = weights_scales,
    target_col       = "sales_quantity",
    price_col        = "sell_price",
)

suites = {
    device: M5ExplorationSuite(
        horizon  = 28,
        ag_path  = "/mnt/lab/nmwamsojo/autogluon_models/explorations",
        base_dir = "/mnt/lab/nmwamsojo/prepared_data",
    )
    for device in DEVICES
}
print(f"Evaluator ready. Suites: {list(suites.keys())}")

# 5 · Store ID Map

In [ ]:
assert "store_id" in static_df.columns, "static_df must contain 'store_id' column"

STORE_IDS: dict[str, set] = {}
for store in STORES:
    items = static_df[static_df["store_id"] == store]["id"].tolist()
    STORE_IDS[store] = set(items)

# GPU assignment map
STORE_DEVICE_MAP: dict[str, str] = {
    store: DEVICES[i % N_GPUS] for i, store in enumerate(STORES)
}

total_items = sum(len(v) for v in STORE_IDS.values())
print(f"{'Store':>6}  {'Device':>8}  {'Items':>7}")
print("-" * 28)
for store in STORES:
    print(f"{store:>6}  {STORE_DEVICE_MAP[store]:>8}  {len(STORE_IDS[store]):>7,}")
print(f"{'TOTAL':>6}  {'':>8}  {total_items:>7,}")

# Sanity: all hist items should belong to a known store
_hist_ids     = set(hist_df_trimmed["id"].unique())
_store_all    = set().union(*STORE_IDS.values())
_unmapped     = _hist_ids - _store_all
if _unmapped:
    print(f"WARNING: {len(_unmapped)} hist items not mapped to any store")
else:
    print(f"\nAll {len(_hist_ids):,} hist items mapped to a store. OK.")

# 6 · Helper Functions & Model Lock

In [ ]:
# Serialises model loads — Accelerate dispatch_model is not thread-safe.
# Cache hits (parquet already exists) bypass this lock entirely.
_model_load_lock = threading.Lock()


def _make_seg_tag(seg_scheme: str, cl: int, ft_steps: int,
                  ft_mode: str, ft_lr: float, ft_bs: int, cov_type: str) -> str:
    if ft_steps == 0:
        return f"hpo_zs_{seg_scheme}_cl{cl}_{cov_type}"
    lr_str = f"{ft_lr:.0e}".replace("-0", "-")
    return (
        f"hpo_ft_{seg_scheme}_cl{cl}"
        f"_steps{ft_steps}_{ft_mode}_lr{lr_str}_bs{ft_bs}_{cov_type}"
    )


def _bools_to_cov_type(use_is_weekend: bool, use_event: bool,
                        use_price: bool, use_snap: bool) -> str:
    """Same logic as v7 — output matches all v4-v6 cov_type strings."""
    parts = []
    if use_is_weekend: parts.append("is_weekend")
    if use_event:      parts.append("event")
    if use_price:      parts.append("price")
    if use_snap:       parts.append("snap")
    return "_".join(parts) if parts else "none"


def _make_store_tag(store_id: str, cl: int, ft_steps: int, ft_mode: str,
                    ft_lr: float, ft_bs: int, cov_type: str) -> str:
    """Store-prefixed cache tag: store_<id>_<base_seg_tag>.

    The store prefix ensures that CA_1 and CA_2 with identical configs
    never share the same parquet file.
    """
    base = _make_seg_tag("all", cl, ft_steps, ft_mode, ft_lr, ft_bs, cov_type)
    return f"store_{store_id.lower()}_{base}"


def _get_store_tag(store_id: str) -> str:
    cfg      = STORE_CONFIGS[store_id]
    cov_type = _bools_to_cov_type(
        cfg["use_is_weekend"], cfg["use_event"],
        cfg["use_price"],      cfg["use_snap"],
    )
    return _make_store_tag(
        store_id, cfg["context_length"], cfg["ft_steps"],
        cfg["ft_mode"], cfg["ft_lr"], cfg["ft_bs"], cov_type,
    )


def _get_forecast_path(store_tag: str) -> str:
    return os.path.join(
        suites[DEVICES[0]].base_dir, DATA_TAG, "level_12",
        CUTOFF_DAY.replace("-", ""), "models", store_tag, "forecasts.parquet",
    )


print("Helpers ready: _make_seg_tag, _bools_to_cov_type, _make_store_tag, _get_store_tag, _get_forecast_path")

# 7 · Cache Status Table

In [ ]:
print(f"{'Store':>6} | {'Mode':>8} | {'Tag':<68} | {'Status':>7} | {'MB':>8} | Action")
print("-" * 118)
for store in STORES:
    cfg     = STORE_CONFIGS[store]
    tag     = _get_store_tag(store)
    path    = _get_forecast_path(tag)
    exists  = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1024**2 if exists else 0.0
    mode    = "ZS" if cfg["ft_steps"] == 0 else f"FT-{cfg['ft_steps']}"
    action  = "skip (cached)" if exists else "will run"
    status  = "exists" if exists else "missing"
    print(f"{store:>6} | {mode:>8} | {tag[:68]:<68} | {status:>7} | {size_mb:>8.1f} | {action}")

# 8 · Store Runner

**Phase 1 — CPU prep** (parallel, `max_workers=4`): filter `hist_df_trimmed`,
`future_df`, and `static_df` to each store's item IDs. This runs on CPU
threads before GPU work starts so that no GPU sits idle waiting for data.

**Phase 2 — GPU execution** (parallel, `max_workers=2`): one store per GPU.
Cache hits bypass `_model_load_lock`; misses serialise the model load.
A psutil RAM check gate each miss — worker pauses 60 s and retries up to 3×
before skipping the store.

After each store: `torch.cuda.empty_cache()` + `gc.collect(2)` + `malloc_trim(0)`.

In [ ]:
def _prepare_store_data(
    store_id: str,
    store_item_ids: set,
) -> tuple:
    """Filter DataFrames to store items. Called from CPU prep thread pool."""
    # FIX 3: id columns are Categorical — filter keeps ALL original categories.
    # value_counts(sort=False) then returns 30490 entries with 27441 zero-count
    # ones, causing iloc OOB crash in df_utils.py.
    hist   = hist_df_trimmed[hist_df_trimmed["id"].isin(store_item_ids)].copy()
    future = future_df[future_df["id"].isin(store_item_ids)].copy()
    if hasattr(hist["id"], "cat"):
        hist["id"]   = hist["id"].cat.remove_unused_categories()
    if hasattr(future["id"], "cat"):
        future["id"] = future["id"].cat.remove_unused_categories()
    static = (
        static_df[static_df["id"].isin(store_item_ids)].copy()
        if static_df is not None else None
    )
    return store_id, hist, future, static


def _check_ram(store_id: str) -> bool:
    """Return True if RAM is below cap, False after exhausting retries."""
    proc = psutil.Process()
    for attempt in range(1, RAM_MAX_RETRIES + 1):
        rss_gb = proc.memory_info().rss / 1024**3
        if rss_gb < RAM_CAP_GB:
            return True
        print(f"  [{store_id}] RAM {rss_gb:.1f} GB > {RAM_CAP_GB} GB cap — "
              f"sleeping {RAM_RETRY_SLEEP}s (attempt {attempt}/{RAM_MAX_RETRIES})")
        time.sleep(RAM_RETRY_SLEEP)
    return False


def _run_one_store(
    store_id:     str,
    store_hist:   pd.DataFrame,
    store_future: pd.DataFrame,
    store_static: "pd.DataFrame | None",
    device:       str,
) -> tuple:
    """
    Fit/infer one store model on the assigned GPU.
    Returns (store_id, forecast_parquet_path) on success,
            (store_id, None) if skipped or failed.
    suite.run() writes the parquet before returning; we del the DataFrame
    immediately to avoid accumulating 10 large objects in memory.
    """
    device_idx = int(device.split(":")[-1]) if ":" in device else 0
    torch.cuda.set_device(device_idx)

    cfg      = STORE_CONFIGS[store_id]
    ft_steps = cfg["ft_steps"]
    ft_mode  = cfg["ft_mode"]  if ft_steps > 0 else "lora"
    ft_lr    = cfg["ft_lr"]    if ft_steps > 0 else 1e-4
    ft_bs    = cfg["ft_bs"]    if ft_steps > 0 else 128
    cl       = cfg["context_length"]
    cov_type = _bools_to_cov_type(
        cfg["use_is_weekend"], cfg["use_event"],
        cfg["use_price"],      cfg["use_snap"],
    )

    store_tag     = _make_store_tag(store_id, cl, ft_steps, ft_mode, ft_lr, ft_bs, cov_type)
    forecast_path = _get_forecast_path(store_tag)
    is_cache_hit  = os.path.exists(forecast_path)
    _lock_ctx     = contextlib.nullcontext() if is_cache_hit else _model_load_lock

    if not is_cache_hit and not _check_ram(store_id):
        print(f"  [{store_id}] RAM cap exceeded after {RAM_MAX_RETRIES} retries — skipping")
        return store_id, None

    exp_cfg = {
        **CFG_CHRONOS_BASE,
        "context_length":       cl,
        "fine_tune_steps":      ft_steps,
        "fine_tune_mode":       ft_mode,
        "fine_tune_lr":         ft_lr,
        "fine_tune_batch_size": ft_bs,
        "batch_size":           BATCH_SIZE,
        "known_cov_cols":       COVARIATE_OPTIONS[cov_type],
        "device":               device,
    }

    try:
        mode_str = "ZS" if ft_steps == 0 else f"FT steps={ft_steps}"
        print(f"  [{store_id}] {mode_str}  cl={cl}  cov={cov_type}  "
              f"device={device}  {'(cache hit)' if is_cache_hit else '(running)'}")

        # Cache hits skip model load; misses serialise load AND clear TSDF cache
        # inside the lock so no other thread can repopulate it between clear/run.
        if is_cache_hit:
            fcst = suites[device].run(
                hist_df      = store_hist,
                future_df    = store_future,
                static_df    = store_static,
                model        = "Chronos2",
                exp_config   = exp_cfg,
                exp_tag      = store_tag,
                data_tag     = DATA_TAG,
                cutoff_day   = CUTOFF_DAY,
                wrapper_dict = WRAPPER,
                force_run    = False,
            )
        else:
            with _model_load_lock:
                suites[device]._cached_tsdf.clear()
                suites[device]._cached_future.clear()
                fcst = suites[device].run(
                    hist_df      = store_hist,
                    future_df    = store_future,
                    static_df    = store_static,
                    model        = "Chronos2",
                    exp_config   = exp_cfg,
                    exp_tag      = store_tag,
                    data_tag     = DATA_TAG,
                    cutoff_day   = CUTOFF_DAY,
                    wrapper_dict = WRAPPER,
                    force_run    = False,
                )

        # suite.run() has already written forecasts.parquet — discard in-memory copy.
        del fcst
        return store_id, forecast_path

    except Exception as e:
        print(f"  [{store_id}] FAILED: {e}")
        return store_id, None

    finally:
        with torch.cuda.device(device_idx):
            torch.cuda.empty_cache()
        gc.collect(2)
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except Exception:
            pass


print("_prepare_store_data, _check_ram, _run_one_store ready.")

_prepare_store_data, _check_ram, _run_one_store ready.


In [ ]:
print("=" * 65)
print("Phase 1: Pre-filtering store DataFrames on CPU threads")
print("=" * 65)

# Clear suite TSDF caches before starting
for _s in suites.values():
    _s._cached_tsdf.clear()
    _s._cached_future.clear()
gc.collect(2)
for _d in DEVICES:
    with torch.cuda.device(_d):
        torch.cuda.empty_cache()
print("Suite caches cleared.\n")

prepared: dict[str, tuple] = {}  # store_id → (hist_df, future_df, static_df)
with ThreadPoolExecutor(max_workers=CPU_PREP_WORKERS,
                         thread_name_prefix="cpu_prep") as prep_pool:
    prep_futs = {
        prep_pool.submit(_prepare_store_data, sid, STORE_IDS[sid]): sid
        for sid in STORES
    }
    for fut in as_completed(prep_futs):
        sid, h, f, s = fut.result()
        prepared[sid] = (h, f, s)
        print(f"  [prep] {sid}: {len(h):,} rows")

print(f"\nAll {len(prepared)} stores prepared.")
print("\n" + "=" * 65)
print("Phase 2: Running store models on GPUs (max 2 concurrent)")
print("=" * 65 + "\n")

completed_paths: dict[str, "str | None"] = {}

with ThreadPoolExecutor(max_workers=MAX_CONCURRENT,
                         thread_name_prefix="gpu_worker") as gpu_pool:
    gpu_futs = {
        gpu_pool.submit(
            _run_one_store,
            store,
            prepared[store][0],
            prepared[store][1],
            prepared[store][2],
            STORE_DEVICE_MAP[store],
        ): store
        for store in STORES
    }
    for fut in as_completed(gpu_futs):
        store = gpu_futs[fut]
        try:
            sid, path = fut.result()
            completed_paths[sid] = path
            status = "OK" if path else "SKIPPED"
            print(f"  [{sid}] {status} → {path or 'N/A'}")
        except Exception as e:
            print(f"  [{store}] EXCEPTION: {e}")
            completed_paths[store] = None

# Free all pre-prepared DataFrames
del prepared
gc.collect(2)
try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass

print("\n" + "=" * 65)
n_ok = sum(1 for v in completed_paths.values() if v)
print(f"Completed: {n_ok}/{len(STORES)} stores")
_skipped = [k for k, v in completed_paths.items() if v is None]
if _skipped:
    print(f"Skipped/failed: {_skipped}")

# 9 · Ensemble Assembly & Evaluation

Load store forecasts **one at a time** from parquet cache.
At most 2 DataFrames are in memory simultaneously (rolling concat pattern).
Equal-weight average per `(id, date)` is trivial here because each item
belongs to exactly one store — concatenation gives the ensemble forecast.
For a true cross-store average (all 10 models predicting all items),
pass `hist_df=hist_df_trimmed` to `_run_one_store` and group-average here.

In [ ]:
print("Assembling ensemble from parquet cache (rolling concat, max 2 DFs in memory)...")
print("-" * 65)

rolling_fcst: "pd.DataFrame | None" = None
skipped_stores: list[str] = []

for store in STORES:
    path = completed_paths.get(store)
    if path is None or not os.path.exists(path):
        print(f"  [{store}] no forecast available — skipping")
        skipped_stores.append(store)
        continue

    raw      = pd.read_parquet(path)
    filtered = raw[raw["id"].isin(STORE_IDS[store])][["id", "date", "mean"]].copy()
    del raw
    gc.collect()
    print(f"  [{store}] {len(filtered):,} rows  ({os.path.getsize(path)/1024**2:.1f} MB)")

    # Rolling concat — keeps at most 2 DFs in memory (previous rolling + new filtered)
    if rolling_fcst is None:
        rolling_fcst = filtered
    else:
        rolling_fcst = pd.concat([rolling_fcst, filtered], ignore_index=True)
        del filtered
        gc.collect()

ensemble_fcst = rolling_fcst if rolling_fcst is not None else pd.DataFrame()
del rolling_fcst
gc.collect(2)

if skipped_stores:
    print(f"\nWARNING: {len(skipped_stores)} stores skipped: {skipped_stores}")

if ensemble_fcst.empty:
    print("ERROR: No store forecasts available — cannot evaluate.")
    ENSEMBLE_WRMSSE = float("nan")
else:
    print(f"\nEnsemble forecast: {ensemble_fcst.shape}")
    print(f"Unique items      : {ensemble_fcst['id'].nunique():,}")
    print(f"Date range        : {ensemble_fcst['date'].min().date()} "
          f"\u2192 {ensemble_fcst['date'].max().date()}")

    metrics = evaluator.evaluate_all(ensemble_fcst, df_actual)
    ENSEMBLE_WRMSSE = float(metrics["WRMSSE"])
    print(f"\nEnsemble WRMSSE = {ENSEMBLE_WRMSSE:.4f}")
    del ensemble_fcst, metrics
    gc.collect(2)

# 10 · Per-Store WRMSSE Breakdown

In [ ]:
print("Computing per-store WRMSSE (loads parquet one at a time)...\n")

per_store_wrmsse: dict[str, float] = {}

for store in STORES:
    path = completed_paths.get(store)
    if path is None or not os.path.exists(path):
        per_store_wrmsse[store] = float("nan")
        continue

    raw        = pd.read_parquet(path)
    store_fcst = raw[raw["id"].isin(STORE_IDS[store])][["id", "date", "mean"]].copy()
    del raw
    try:
        m = evaluator.evaluate_all(store_fcst, df_actual)
        per_store_wrmsse[store] = float(m["WRMSSE"])
        del m
    except Exception as e:
        print(f"  [{store}] evaluation error: {e}")
        per_store_wrmsse[store] = float("nan")
    del store_fcst
    gc.collect()

# Display summary table
print(f"\n{'Store':>6} | {'Device':>8} | {'Mode':>8} | {'WRMSSE':>8}")
print("-" * 42)
for store in STORES:
    cfg      = STORE_CONFIGS[store]
    device   = STORE_DEVICE_MAP[store]
    mode     = "ZS" if cfg["ft_steps"] == 0 else f"FT-{cfg['ft_steps']}"
    w        = per_store_wrmsse[store]
    w_str    = f"{w:.4f}" if not (w != w) else "N/A"  # nan check
    print(f"{store:>6} | {device:>8} | {mode:>8} | {w_str:>8}")
print("-" * 42)
print(f"{'Ensemble':>6} | {'':>8} | {'':>8} | {ENSEMBLE_WRMSSE:>8.4f}")

# Pandas summary for easy comparison across runs
df_results = pd.DataFrame([
    {
        "store":   store,
        "device":  STORE_DEVICE_MAP[store],
        "ft_steps": STORE_CONFIGS[store]["ft_steps"],
        "cov_type": _bools_to_cov_type(
            STORE_CONFIGS[store]["use_is_weekend"],
            STORE_CONFIGS[store]["use_event"],
            STORE_CONFIGS[store]["use_price"],
            STORE_CONFIGS[store]["use_snap"],
        ),
        "WRMSSE":  per_store_wrmsse[store],
    }
    for store in STORES
])
display(
    df_results.style
    .format({"WRMSSE": "{:.4f}"})
    .background_gradient(subset=["WRMSSE"], cmap="RdYlGn_r")
    .set_caption(f"Per-store WRMSSE  |  Ensemble WRMSSE = {ENSEMBLE_WRMSSE:.4f}")
)

# 11 · Extension: Store × Category / Store × Department

Commented skeleton for scaling to 30 (store×cat) or 70 (store×dept) models.
Follow the same pattern — only the inner loop and tag prefix change.

In [ ]:
# ── Store × Category extension (30 models) ───────────────────────────────────
# Uncomment and populate STORE_CAT_CONFIGS with per-(store, cat) dicts.
#
# CATEGORIES = ["FOODS", "HOBBIES", "HOUSEHOLD"]
#
# STORE_CAT_IDS: dict[tuple, set] = {}
# for store in STORES:
#     for cat in CATEGORIES:
#         mask = (
#             (static_df["store_id"] == store) &
#             (static_df["cat_id"]   == cat)
#         )
#         STORE_CAT_IDS[(store, cat)] = set(static_df[mask]["id"].tolist())
#
# def _make_store_cat_tag(store, cat, cl, ft_steps, ft_mode, ft_lr, ft_bs, cov_type):
#     base = _make_seg_tag("all", cl, ft_steps, ft_mode, ft_lr, ft_bs, cov_type)
#     return f"store_{store.lower()}_{cat.lower()}_{base}"
#
# Then feed (store_hist & cat_hist filtered jointly) into the same
# _run_one_store / ThreadPoolExecutor structure above.


# ── Store × Department extension (70 models) ──────────────────────────────────
# DEPARTMENTS = [
#     "FOODS_1", "FOODS_2", "FOODS_3",
#     "HOBBIES_1", "HOBBIES_2",
#     "HOUSEHOLD_1", "HOUSEHOLD_2",
# ]
#
# STORE_DEPT_IDS: dict[tuple, set] = {}
# for store in STORES:
#     for dept in DEPARTMENTS:
#         mask = (
#             (static_df["store_id"] == store) &
#             (static_df["dept_id"]  == dept)
#         )
#         STORE_DEPT_IDS[(store, dept)] = set(static_df[mask]["id"].tolist())
#
# def _make_store_dept_tag(store, dept, cl, ft_steps, ft_mode, ft_lr, ft_bs, cov_type):
#     base = _make_seg_tag("all", cl, ft_steps, ft_mode, ft_lr, ft_bs, cov_type)
#     return f"store_{store.lower()}_{dept.lower().replace('_', '')}_{base}"
#
# With 70 models and 2 GPUs, schedule them in two rounds of 35.
# Adjust MAX_CONCURRENT if you add more GPUs.
print("Extension skeletons available above (uncomment to use).")